# Microsoft Retriever 1 (preview): Complete Answers to Multi-Part Questions

Real questions often have several parts: *"Who performed the first spacewalk, **on which mission**, and **which cosmodrome** did it launch from?"* A single search query rarely finds evidence for every part, so the agent's answer comes back with gaps, or worse, guesses.

**Microsoft Retriever 1** (`microsoft-retriever-1`) is a retrieval model **hosted inside Foundry IQ** that plans the searches for you: it reads the full question, decides which searches to run, and merges the evidence. You don't connect, authenticate or pay for an external chat model just to plan retrieval.

**What you'll see in this notebook**

| | Plain hybrid search | With Microsoft Retriever 1 |
|---|---|---|
| *First spacewalk: who, which mission, which cosmodrome?* | Typical answer: *"Leonov, Voskhod 2. The cosmodrome is **not in sources**."* | *"Leonov, Voskhod 2, launched from **Baikonur Cosmodrome**."* |
| Facts in the answer, across four easy-to-hard questions | ~75% | ~95% (Retriever 1 knowledge base) |
| Versus planning with your own LLM (`medium` effort) | | As many facts or more, in **a fraction of the time**, with no external model |

> Your exact numbers will differ slightly; the notebook measures them live.

You'll use Retriever 1 three ways:

1. **Direct agentic search**: `query_type=QueryType.AGENTIC` on an ordinary index query, no knowledge base needed
2. **A knowledge base that uses only Retriever 1**, with `minimal` reasoning effort
3. **`auto` reasoning effort**: Retriever 1 handles what it can and escalates to your external model only when a request needs deeper planning

![Three ways to use Microsoft Retriever 1: direct agentic search, a Retriever 1-only knowledge base at minimal effort, and auto effort that escalates to an external model only when needed](../docs/images/retriever-1.svg)

## Prerequisites

| Requirement | Notes |
|---|---|
| [prepare-dataset.ipynb](prepare-dataset.ipynb) completed | Provides the `space-exploration` index |
| Search service on the **Standard Knowledge Retrieval** plan | Required for Retriever 1. See [Enable or disable agentic retrieval billing](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-enable-disable) |
| Search service in a [region that supports Microsoft Retriever 1](https://learn.microsoft.com/azure/search/search-region-support#microsoft-retriever-1): **Japan East, North Europe, West Central US or West US 3** (as of November 2026) | Elsewhere, agentic queries fail with *"Agentic retrieval is not available in this region"* |
| Roles: **Search Service Contributor**, **Search Index Data Reader** | Create the knowledge source and knowledge bases, and query |
| An LLM deployment (`LLM_DEPLOYMENT`) | For the `auto` section (the search identity needs **Cognitive Services User** on the Foundry resource) and to write the example answers (**you** need **Cognitive Services OpenAI User**) |

> **Billing:** Retriever 1 is billed on **input tokens** processed during query planning (output tokens are free). This is separate from Foundry IQ retrieval tokens and from Azure OpenAI charges. See [Use Microsoft Retriever 1](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-use-microsoft-retriever).


## Setup

In [ ]:
# %pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv requests matplotlib


In [ ]:
import json
import os
import re
import time

import matplotlib.pyplot as plt
import requests
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model,
    KnowledgeSourceReference,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalAutoReasoningEffort,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalMediumReasoningEffort,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)
from azure.search.documents.models import AgenticSearchOptions, QueryDebugMode, QueryType, VectorizableTextQuery
from dotenv import load_dotenv

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")
credential = DefaultAzureCredential()
search_credential = AzureKeyCredential(search_api_key) if search_api_key else credential

ai_services_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].split("/api/projects")[0]
llm_deployment = os.environ["LLM_DEPLOYMENT"]
llm_model = os.environ["LLM_MODEL"]

INDEX_NAME = "space-exploration"
KNOWLEDGE_SOURCE = "space-mr1-ks"
KB_RETRIEVER_ONLY = "space-mr1-kb"
KB_AUTO = "space-mr1-auto-kb"

index_client = SearchIndexClient(search_endpoint, search_credential)
search_client = SearchClient(search_endpoint, INDEX_NAME, search_credential)

print(f"Search endpoint:                 {search_endpoint}")
print(f"External model ('auto', answers): {llm_deployment} / {llm_model}")


## Step 1: The problem, and the fix, in one question

**What we're searching:** the `space-exploration` index from [prepare-dataset](prepare-dataset.ipynb), about 4,000 chunks from ~230 Wikipedia articles.

We ask one multi-part question two ways and hand the top 5 chunks from each to the **same chat model** with the **same prompt**. The model is told to say *"not in sources"* for any part it can't support, so gaps in the evidence show up as gaps in the answer.

1. **Plain hybrid search** (keyword + vector + semantic ranking): one query for the whole question
2. **Direct agentic search** with Retriever 1: set the query type to `agentic` and pass the full question. Retriever 1 decides which searches to run. The debug info shows:
   - `sub_queries`: the searches it issued
   - `termination_reason`: `modelStopped` (finished normally), `maxRuntimeExceeded` or `modelFailed`
   - `usage`: planning and reasoning tokens

A short **fact checklist** scores each answer.

### Helpers


In [ ]:
def hybrid_search(question: str, top: int = 5) -> list[dict]:
    results = search_client.search(
        search_text=question,
        vector_queries=[VectorizableTextQuery(text=question, k_nearest_neighbors=50, fields="text_vector")],
        query_type=QueryType.SEMANTIC,
        select=["title", "chunk"],
        top=top,
    )
    return list(results)


def agentic_search(question: str, top: int = 5, filter_expr: str | None = None):
    """Return (documents, agentic debug info) for a Retriever 1 agentic query."""
    results = search_client.search(
        search_text=question,
        query_type=QueryType.AGENTIC,
        agentic_options=AgenticSearchOptions(max_runtime_in_seconds=60),
        filter=filter_expr,
        select=["title", "chunk"],
        top=top,
        debug=QueryDebugMode.AGENTIC,
    )
    documents = list(results)
    debug = results.get_debug_info()
    return documents, (debug.agentic if debug else None)


def write_answer(question: str, chunks: list[dict]) -> str:
    """Have the chat deployment answer from the given chunks only, flagging unsupported parts."""
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    sources = "\n\n".join(f"[{c['title']}] {c['chunk'][:1200]}" for c in chunks)
    response = requests.post(
        f"{ai_services_endpoint}/openai/deployments/{llm_deployment}/chat/completions?api-version=2024-10-21",
        headers={"Authorization": f"Bearer {token}"},
        json={"messages": [
            {"role": "system", "content": "Answer only from the sources, in 2-4 sentences. "
                                          "If part of the question isn't covered by the sources, say 'not in sources' for that part."},
            {"role": "user", "content": f"Question: {question}\n\nSources:\n{sources}"},
        ]},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def score(text: str, facts: dict) -> list[str]:
    return [fact for fact, pattern in facts.items() if re.search(pattern, text)]


def show_answer(label: str, question: str, chunks: list[dict], facts: dict) -> None:
    answer = write_answer(question, chunks)
    found = score(answer, facts)
    missing = [f for f in facts if f not in found]
    print(f"{'═' * 15} {label}: {len(found)}/{len(facts)} facts {'═' * 15}")
    print(f"Evidence from: {', '.join(dict.fromkeys(c['title'] for c in chunks))}\n")
    print(answer)
    print(f"\n✔ {', '.join(found) or '—'}" + (f"    ✘ missing: {', '.join(missing)}" if missing else "") + "\n")


def print_agentic_debug(agentic) -> None:
    if agentic is None:
        print("No agentic debug info returned")
        return
    reason = getattr(agentic.termination_reason, "value", agentic.termination_reason)
    print(f"Stopped because:  {reason}")
    if agentic.usage:
        print(f"Tokens used:      planning {agentic.usage.agentic_planning_tokens:,}, reasoning {agentic.usage.reasoning_tokens:,}")
    for sub_query in agentic.sub_queries or []:
        print(f"  subquery (iteration {sub_query.iteration}): {sub_query.search}")


### Run it

Three facts are needed: **who** (Alexei Leonov), **which mission** (Voskhod 2) and **which cosmodrome** (Baikonur). The cosmodrome lives in a different part of the corpus from the spacewalk itself, which is exactly what a single query tends to miss.


In [ ]:
QUESTION = "Who performed the first spacewalk, on which mission, and which cosmodrome did that mission launch from?"
QUESTION_FACTS = {"Alexei Leonov": r"Leonov", "Voskhod 2": r"Voskhod 2", "Baikonur": r"Baikonur"}

show_answer("Plain hybrid search", QUESTION, hybrid_search(QUESTION), QUESTION_FACTS)


In [ ]:
start = time.perf_counter()
documents, agentic = agentic_search(QUESTION)
elapsed = time.perf_counter() - start

show_answer(f"Agentic search with Microsoft Retriever 1 ({elapsed:.1f}s)", QUESTION, documents, QUESTION_FACTS)
print("How Retriever 1 searched:")
print_agentic_debug(agentic)


Filters still apply, and they constrain all of the evidence Retriever 1 is allowed to use. Here we restrict it to the Soviet programme articles.

> Direct agentic search doesn't support `skip`, `order_by`, `include_total_count`, facets, highlighting, query rewrites, speller, semantic answers or captions.

In [ ]:
soviet_titles = "Voskhod 2,Alexei Leonov,Baikonur Cosmodrome,Vostok 1,Yuri Gagarin,Sergei Korolev"
documents, agentic = agentic_search(QUESTION, filter_expr=f"search.in(title, '{soviet_titles}', ',')")
print(f"Articles used (all inside the filter): {', '.join(sorted({doc['title'] for doc in documents}))}")
print_agentic_debug(agentic)


## Step 2: A knowledge base that uses only Retriever 1

Knowledge bases add citations, an activity log, multi-source support and an MCP endpoint for agents. To use Retriever 1 in a knowledge base:

- Set `models` to `[KnowledgeBaseMicrosoftRetriever1Model()]`
- Use **`minimal`** reasoning effort. A Retriever 1-only knowledge base doesn't support `low` or `medium`, or knowledge-source selection
- Send **exactly one semantic intent** with the complete question. Multiple intents skip Retriever 1 and return `206` with a warning

### Create the knowledge source and knowledge base

In [ ]:
knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE,
    description="Wikipedia chunks about space exploration: missions, astronauts, agencies, launch sites and spacecraft.",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        source_data_fields=[SearchIndexFieldReference(name=f) for f in ("title", "source_url", "chunk")],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source)
print(f"Knowledge source '{KNOWLEDGE_SOURCE}' created or updated")

In [ ]:
kb_retriever_only = KnowledgeBase(
    name=KB_RETRIEVER_ONLY,
    description="Space exploration knowledge base planned entirely by Microsoft Retriever 1.",
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    models=[KnowledgeBaseMicrosoftRetriever1Model()],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(kb_retriever_only)
print(f"Knowledge base '{KB_RETRIEVER_ONLY}' created or updated")

### Retrieval helpers

`retrieve()` sends one semantic intent and optionally overrides the reasoning effort for that request. The other helpers read the **activity log** (which models planned the request, tokens, timing) and the **references** (which articles supplied evidence).

In [ ]:
EFFORTS = {
    "minimal": KnowledgeRetrievalMinimalReasoningEffort,
    "low": KnowledgeRetrievalLowReasoningEffort,
    "medium": KnowledgeRetrievalMediumReasoningEffort,
    "auto": KnowledgeRetrievalAutoReasoningEffort,
}


def retrieve(kb_name: str, question: str, effort: str | None = None):
    """Return (response, elapsed seconds). `effort` overrides the KB default for this request only."""
    client = KnowledgeBaseRetrievalClient(search_endpoint, search_credential, knowledge_base_name=kb_name)
    request = KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=question)],
        retrieval_reasoning_effort=EFFORTS[effort]() if effort else None,
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=[SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE,
            include_references=True,
            include_reference_source_data=True,
        )],
    )
    start = time.perf_counter()
    response = client.retrieve(request)
    return response, time.perf_counter() - start


def planning_models(response) -> list[str]:
    return [
        record.model.model_name
        for record in response.activity or []
        if record.type == "modelQueryPlanning" and getattr(record, "model", None)
    ]


def reference_titles(response) -> set[str]:
    return {(ref.source_data or {}).get("title") for ref in response.references or []} - {None}


def show_retrieval(response, elapsed: float) -> None:
    print(f"Elapsed: {elapsed:.1f}s\n")
    print(f"  {'step':<22} {'model':<22} {'tokens in/out':<16} time")
    for record in response.activity or []:
        model = getattr(getattr(record, "model", None), "model_name", "") or "-"
        tokens = f"{record.input_tokens:,}/{record.output_tokens:,}" if getattr(record, "input_tokens", None) is not None else "-"
        elapsed_ms = f"{record.elapsed_ms} ms" if record.elapsed_ms else "-"
        print(f"  {record.type:<22} {model:<22} {tokens:<16} {elapsed_ms}")
        if record.error:
            print(f"    error: {record.error}")
    titles = sorted(reference_titles(response))
    print(f"\nEvidence from {len(titles)} articles: {', '.join(titles)}")


### Run it

In [ ]:
response, elapsed = retrieve(KB_RETRIEVER_ONLY, QUESTION)
show_retrieval(response, elapsed)

Look for a `modelQueryPlanning` record with model `microsoft-retriever-1`: that confirms Retriever 1 planned the request. If it's missing, check the prerequisites (plan, region) and that you sent a single intent.

With `extractiveData` output, the knowledge base returns the grounding chunks themselves (no generated answer). This is what an agent or your own LLM call would receive:

In [ ]:
grounding = json.loads(response.response[0].content[0].text)
print(f"{len(grounding)} grounding chunks returned. First 3:\n")
for item in grounding[:3]:
    print(f"[ref_id {item['ref_id']}] {item['title']}")
    print(f"  {' '.join(item['content'].split())[:250]}…\n")


## Step 3: `auto` reasoning effort

Requests vary in difficulty. With `auto`, the knowledge base picks the depth per request:

1. **Initial pass** (minimal): a fast lookup. If the top result is relevant enough, it returns immediately
2. **Retriever 1** runs deeper retrieval if the first pass isn't enough
3. **External model, medium effort**: Retriever 1 hands planning to your model when the request needs it

`auto` **requires an external model** on the knowledge base, so this knowledge base lists both models. Requests with more than one intent, or knowledge bases with sources that don't support the initial pass (for example MCP or web), start at medium instead.

In [ ]:
kb_auto = KnowledgeBase(
    name=KB_AUTO,
    description="Space exploration knowledge base with auto reasoning effort.",
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=ai_services_endpoint, deployment_name=llm_deployment, model_name=llm_model,
        )),
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalAutoReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(kb_auto)
print(f"Knowledge base '{KB_AUTO}' created or updated")

### Run an easy and a hard question

In [ ]:
EASY = "When did Sputnik 1 launch?"

response, elapsed = retrieve(KB_AUTO, EASY)
show_retrieval(response, elapsed)
print(f"Planning models used: {', '.join(planning_models(response)) or 'none (the initial pass was enough)'}")


In [ ]:
HARD = (
    "Compare how the Soviet, American and Chinese programmes each put their first person into orbit: "
    "who flew, on which spacecraft, in what year, and from which launch site?"
)

response, elapsed = retrieve(KB_AUTO, HARD)
show_retrieval(response, elapsed)
print(f"Planning models used: {', '.join(planning_models(response)) or 'none (the initial pass was enough)'}")


For the easy question you should see little or no model planning: the initial pass was sufficient. For the hard question you'll typically see `microsoft-retriever-1`, and possibly a second `modelQueryPlanning` record for your external model, which means Retriever 1 escalated.

## Step 4: Scoreboard — answer quality vs speed

Now put all the paths side by side on several questions, from easy to hard. For each path we take its evidence, have the **same chat model** write an answer with the **same prompt**, and score the answer against a **fact checklist**. We also time the retrieval.

| Path | Who plans the searches |
|---|---|
| `hybrid` | Nobody: one query, top 5 chunks |
| `agentic` | Microsoft Retriever 1 (direct agentic search, top 5 chunks) |
| `kb-mr1` | Knowledge base, `minimal` effort, Retriever 1 only |
| `kb-auto` | Knowledge base, `auto` effort: initial pass → Retriever 1 → your model if needed |
| `kb-medium` | Knowledge base, `medium` effort: your external model plans everything |

Knowledge base answers use the first 10 chunks returned, as an agent would.

### Helpers


In [ ]:
def kb_chunks(kb: str, effort: str | None = None):
    def run(question: str):
        response, _ = retrieve(kb, question, effort)
        chunks = [dict(ref.source_data or {}) for ref in response.references or []][:10]
        return chunks, ", ".join(dict.fromkeys(planning_models(response))) or "-"
    return run


PATHS = {
    "hybrid": lambda q: (hybrid_search(q), "-"),
    "agentic": lambda q: (agentic_search(q)[0], "microsoft-retriever-1"),
    "kb-mr1": kb_chunks(KB_RETRIEVER_ONLY),
    "kb-auto": kb_chunks(KB_AUTO),
    "kb-medium": kb_chunks(KB_AUTO, "medium"),
}


def run_scoreboard(questions: dict) -> list[dict]:
    rows = []
    print(f"{'question':<44} {'path':<10} {'secs':>5} {'facts':>6}  planners")
    for question, facts in questions.items():
        for path, run in PATHS.items():
            start = time.perf_counter()
            chunks, planners = run(question)
            secs = time.perf_counter() - start
            found = score(write_answer(question, chunks), facts)
            rows.append({"question": question, "path": path, "secs": secs, "found": len(found), "total": len(facts)})
            label = question[:42] if path == "hybrid" else ""
            print(f"{label:<44} {path:<10} {secs:>5.1f} {len(found):>3}/{len(facts):<2}  {planners}")
        print()
    return rows


def plot_scoreboard(rows: list[dict]) -> None:
    paths = list(PATHS)
    coverage = [100 * sum(r["found"] for r in rows if r["path"] == p) / sum(r["total"] for r in rows if r["path"] == p) for p in paths]
    latency = [sum(r["secs"] for r in rows if r["path"] == p) / len({r["question"] for r in rows}) for p in paths]
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.8))
    left.bar(paths, coverage, color=["#9AA3B5", "#7B5BD6", "#7B5BD6", "#3B6FD8", "#D98E1C"])
    left.set_title("Facts in the answer (all questions)")
    left.set_ylabel("%")
    left.set_ylim(0, 105)
    right.bar(paths, latency, color=["#9AA3B5", "#7B5BD6", "#7B5BD6", "#3B6FD8", "#D98E1C"])
    right.set_title("Average retrieval time")
    right.set_ylabel("seconds")
    plt.tight_layout()
    plt.show()


### Run it

In [ ]:
SCOREBOARD = {
    EASY: {"4 October 1957": r"4 October 1957|October 4, 1957"},
    QUESTION: QUESTION_FACTS,
    "Which space telescopes observe in infrared, and where in space is each one positioned?": {
        "James Webb": r"Webb|JWST", "Webb at L2": r"L2|Lagrange", "Spitzer": r"Spitzer",
        "Spitzer's Earth-trailing orbit": r"Earth-trailing|heliocentric", "Herschel": r"Herschel",
    },
    HARD: {
        "Gagarin": r"Gagarin", "Vostok 1": r"Vostok 1", "1961": r"1961", "Baikonur": r"Baikonur",
        "Glenn": r"Glenn", "Friendship 7": r"Friendship 7", "1962": r"1962", "Cape Canaveral": r"Canaveral",
        "Yang Liwei": r"Yang Liwei", "Shenzhou 5": r"Shenzhou 5", "2003": r"2003", "Jiuquan": r"Jiuquan",
    },
}

rows = run_scoreboard(SCOREBOARD)
plot_scoreboard(rows)


### What to take away

In our run, answers built on plain hybrid search contained about **75%** of the checklist facts; the Retriever 1-only knowledge base reached about **95%**, and `medium` (your own LLM planning) about **85%** while taking roughly **4× longer**. Individual rows move a little between runs because the answering model isn't deterministic, so read the averages in the chart rather than any single row.

- **Single queries leave gaps** on multi-part questions: the answer says *"not in sources"* where it should give a fact
- **Retriever 1 closes most of them without an external model.** Direct agentic search is as fast as hybrid search; the Retriever 1 knowledge base returns more evidence (about 20 chunks) and finds the most facts
- **`medium` costs more for no gain here.** It spends model tokens and several seconds on every request
- **`auto` gives you both.** Easy questions stop at the fast initial pass (no planner in the table), harder ones go to Retriever 1, and only the hardest escalate to your model
- **Some facts simply aren't in the corpus** (for example, a launch site the articles never state). No retrieval strategy can fix that; *"not in sources"* is the honest outcome

## When should you use Microsoft Retriever 1?

| Use it for | Consider something else |
|---|---|
| Multi-part questions (*who, which, where*) whose parts live in different chunks | Single-fact lookups: plain hybrid search is enough and cheapest |
| Agents that need citations and an activity log, without managing a planning model (`minimal` KB) | Broad *"give me an overview"* questions: add a [topic graph](topic-graph.ipynb) for breadth |
| Mixed traffic: use `auto` so only hard requests pay for deeper planning | Questions that follow relationships across many hops: use an [entity graph](entity-graph.ipynb) |

## Troubleshooting

| Symptom | Fix |
|---|---|
| No `microsoft-retriever-1` planning record | Confirm the Standard Knowledge Retrieval plan and a supported region; send exactly one semantic intent |
| `206` with an `agenticReasoning` warning | Multiple intents were sent, so Retriever 1 was skipped |
| `modelQueryPlanning` record contains an `error` | Retriever 1 started but didn't finish. With `minimal`, retained results are returned as `206`; with `auto`, planning is handed to the external model |
| `low`/`medium` rejected on the Retriever 1-only KB | Expected: add an external model (as in the `auto` KB) to use them |

## Clean up

Deletes the knowledge bases and knowledge source created here. The `space-exploration` index is kept for the other notebooks.


In [ ]:
# index_client.delete_knowledge_base(KB_AUTO)
# index_client.delete_knowledge_base(KB_RETRIEVER_ONLY)
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE)
# print("Deleted knowledge bases and knowledge source")

## Summary

- **The problem:** one search query rarely covers every part of a multi-part question, so the agent's answer has gaps
- **The fix:** Microsoft Retriever 1, hosted in Foundry IQ, plans several searches from the full question. Answers include facts a single query missed, with no external planning model
- **Three ways to use it:** direct agentic search (`QueryType.AGENTIC`, no knowledge base), a Retriever 1-only knowledge base (`minimal`, citations and activity log), or `auto` (fast initial pass → Retriever 1 → your model only when needed)
- **The trade-off:** on these questions Retriever 1 found as many facts as `medium` effort or more, in a fraction of the time

### Next steps

- [topic-graph.ipynb](topic-graph.ipynb): broad questions about themes across the corpus
- [entity-graph.ipynb](entity-graph.ipynb): multi-hop questions that follow relationships between entities
